# **Access and analyze pre-calculated climate indicators for an ensemble of climate simulations**

This notebook shows you how to access and analyze pre-calculated climate indicators for climate model ensembles such as `CanDCS-M6` ([climatedata.ca](https://climatedata.ca/resource/intro-to-candcs-m6/)). You will learn how to filter the data by scenario and variable, and how to subset them to a region. The notebook then applies through common processing steps, including computing 30-year climatological means, anomalies/deltas relative to a reference period, spatial averages, and ensemble percentiles. The processed results are saved to Zarr files and also exported to CSV for downstream analysis. You can explore and visualize the processed data directly in this notebook or download the CSV file and open it in tools such as Excel.

The example in **Step 6** illustrates how to visualize the seasonal indicator data extracted in the previous steps. The default data selections in **Step 3** and **Step 4** are configured to support this example. You can modify the code in **Steps 3, 4 and 6** to adapt the workflow to your own variables, regions, scenarios, and periods of interest. 

You can also upload your own polygon files (for example, your own shapefiles or GeoJSONs) and modify the `polygon_file` path in **Step 3** accordingly to run the same analysis for your own regions.
<div class="alert alert-warning">
<strong> INFO :</strong> As this tutorial writes data to disk, you must copy the notebook and all associated files to your <strong>writable-workspace</strong> folder. Make sure to re-open the copied version of notebook to ensure that code is running from a location with write permissions  
</div>

### **Preparation – Imports & configuration**

Run this cell once at the very beginning of the notebook. You don't need to modify anything inside it. It just loads the Python packages, defines some helper functions and sets some technical options.


In [ ]:
import logging
import shutil
import warnings
from pathlib import Path

import geopandas as gpd
import hvplot.pandas
import hvplot.xarray
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import panel as pn
import threddsclient as tds
import xarray as xr
import yaml
from dask.diagnostics import ProgressBar
from dask.distributed import Client
from IPython.display import Markdown, clear_output
from xclim.core.units import convert_units_to
from xscen import ensembles as xens
from xscen.aggregate import climatological_op, compute_deltas, spatial_mean
from xscen.io import save_to_zarr
from xscen.spatial import subset
from xscen.utils import unstack_dates

warnings.simplefilter("ignore")
pn.extension("tabulator")

# Dask cluster configuration used when writing Zarr files
dask_kwargs = dict(
    n_workers=6,
    threads_per_worker=10,
    memory_limit="3GB",
    dashboard_address=8787,
    silence_logs=logging.ERROR,
)


# user shapefiles are often very complex and can cause issues with spatial averaging.
# This helper function simplifies the geometries while preserving topology, which should help avoid errors and speed up processing.
def simplify_shape(polygon_file):
    if isinstance(polygon_file, str):
        polygon_file = Path(polygon_file)
    outpoly = Path(polygon_file)
    outpoly = outpoly.parent.joinpath(f"{polygon_file.stem}_simplified").with_suffix(
        ".geojson"
    )
    if not outpoly.exists():
        poly = gpd.read_file(polygon_file, encoding="utf-8").to_crs(epsg=4326)
        nrecs = len(poly)
        # try to repair geometries
        poly["id"] = poly.index
        # tiny buffer
        poly["geometry"] = poly.buffer(0.0001)
        # dissolve polygons on buffered result
        poly = poly.dissolve(by="id").reset_index()
        # apply make_valid()
        poly["geometry"] = poly["geometry"].make_valid()

        # simplify geometries to make spatial avg faster and avoid any other bugs with shapefile
        poly["geometry"] = poly.simplify(tolerance=0.0005, preserve_topology=True)
        assert len(poly) == nrecs
        poly.to_file(outpoly, driver="GeoJSON")
    poly = gpd.read_file(outpoly)
    return poly


def get_sample_data(gdf, data_source=None):
    if data_source == "CanDCS-M6":
        url = "https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/bias_adjusted/cmip6/climatedata_ca/CanDCS-M6/ensemble_statistics/30yAvg/annual_CanDCS-M6_climindices_ensemble_percentiles_30yAvg.ncml"
        chunks = dict(time=1, lon=-1, lat=-1)
    else:
        raise ValueError(f"data_source {data_source} not implemented")

    bbox = gdf.total_bounds

    lon_bnds = [bbox[0] - 0.1, bbox[2] + 0.1]
    lat_bnds = [bbox[1] + 0.1, bbox[3] - 0.1]
    ds_tmp = xr.open_dataset(url, chunks=chunks, decode_timedelta=False).isel(time=-2)
    keep_var = [v for v in ds_tmp.data_vars][0]
    ds_tmp = subset(
        ds_tmp.drop_vars([v for v in ds_tmp.data_vars if v != keep_var]),
        method="bbox",
        lat_bnds=lat_bnds,
        lon_bnds=lon_bnds,
    )
    return ds_tmp[keep_var]


print("Setup done. You can move to Step 1.")

### **Step 1 – Choose the dataset**

First, specify which ensemble of simulations to use for the pre-calculated climate indicators. For this tutorial, we work with the `CanDCS-M6` ensemble from climatedata.ca, hosted on the PAVICS / [THREDDS server](https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/catalog/catalog.html).

This notebook is specifically designed to work with the `CanDCS-M6` dataset, but it can be adapted to other ensembles (e.g., `ESPO-G6-R2`, which is the data behind Ouranos' Climate Portraits site). If you choose a different ensemble, some parts of the code below will need to be adjusted to match that dataset’s structure. 

For the current analysis, you don't need to make any changes in the cell below.

In [ ]:
# this dictionary specifies the THREDDS catalog URL where the data for CanDCS-M6 can be found, and
# a preferred chunking strategy used when loading the data with Dask (this helps performance and memory).
pavics_url = {
    "CanDCS-M6": {
        "url": "https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/catalog/datasets/simulations/bias_adjusted/cmip6/climatedata_ca/CanDCS-M6/ensemble_members/timeseries/catalog.html",
        "chunks": dict(realization=1, time=-1, lon=30 * 5, lat=30 * 5),
    },
}

# Specify which dataset to use for the analysis.
# If you change this, make sure to add the corresponding THREDDS catalog URL in the `pavics_url` dictionary above
# and adjust the rest of the notebook to match the structure of the new dataset.
data_source = "CanDCS-M6"

print("Selected data source:", data_source)

### **Step 2 – List available climate indicators for this dataset**

This step connects to the PAVICS / THREDDS catalog and builds a table of available precalculated climate indicators for the selected dataset. The table includes a column for:

- variable IDs
- which temporal groupings the variables exist for (`annual`, `seasonal`, `monthly`, …)
- long name of the variable
- a description of the variable

No need to modify the code; just scroll and browse the table to decide what variable you want to extract in Step 2. 

In [ ]:
url = pavics_url[data_source]["url"]
chunks = pavics_url[data_source]["chunks"]

vars_dict = {}

print("Scanning the THREDDS catalog…")

for dd in tds.crawl(url, depth=100):
    # Only keep entries belonging to this data source
    if data_source not in dd.name:
        continue

    ds_tmp = xr.open_dataset(dd.opendap_url(), chunks=chunks, decode_timedelta=False)

    # set non-time variables as coordinates
    for vv in ds_tmp.data_vars:
        if "time" not in ds_tmp[vv].dims:
            ds_tmp = ds_tmp.assign_coords({vv: ds_tmp[vv]})

    # Try to infer the time frequency and convert to an interpretable label
    freq = xr.infer_freq(ds_tmp.time)
    if freq is None:
        continue

    freq = (
        freq.replace("YS", "annual")
        .replace("QS-DEC", "seasonal")
        .replace("2QS-OCT", "6month")
        .replace("MS", "monthly")
        .replace("annual-JAN", "annual")
        .replace("annual-JUL", "annual-julyjune")
    )

    # For CanDCS-M6, variables are often named like "ssp245_tx_mean".
    # We remove the scenario prefix so the variable ID is just "tx_mean".
    for vv in [v for v in ds_tmp.data_vars if "_consensus" not in v]:
        vv_out = vv if "ssp" not in vv else "_".join(vv.split("_")[1:])

        if vv_out not in vars_dict:
            vars_dict[vv_out] = {
                "temporal grouping": [freq],
                "description": ds_tmp[vv].attrs.get("long_name", ""),
            }
        else:
            if freq not in vars_dict[vv_out]["temporal grouping"]:
                vars_dict[vv_out]["temporal grouping"].append(freq)

vars_table = pd.DataFrame.from_dict(vars_dict, orient="index").sort_index()
file_path = "climatedata_vars.yml"
with open(file_path) as file:
    # Load the YAML content into a Python dictionary
    config_data = yaml.safe_load(file)
config_data["variables"]
vars_table["climatedata.ca name"] = "N/A"
for kk, vv in config_data["variables"].items():
    vars_table.at[vv, "climatedata.ca name"] = kk.replace("_", " ").capitalize()
vars_table = vars_table[["temporal grouping", "climatedata.ca name", "description"]]
pn.Column(
    pn.pane.Markdown(
        f"## Available precalculated climate indicators for {data_source}"
    ),
    pn.widgets.Tabulator(
        vars_table,
        header_filters=True,
        sortable=True,
        page_size=10,
        disabled=True,
        height=350,
    ),
)

### **Step 3 – Choose the variable, region, and calculation settings**

Next, specify which variable, region and settings you want to use for data extraction and analysis. 

Choose the variable(s) to extract and their corresponding temporal grouping from the table above. Note that clicking a variable in the table does not automatically select it. You must manually enter the temporal grouping and variable ID in the `variables` dictionary below.

- `polygon_file` : the path to the shapefile / GeoJSON defining your region
- `variables` : a dictionary whose keys are temporal frequencies ("seasonal", "monthly", "annual", etc.) and whose values are lists of variable IDs available at those frequencies, as shown in the table above
- `ssps` : emission scenarios you want to analyze (e.g. `"ssp126"`, `"ssp245"`, `"ssp370"`)
- `spatial_avg` : set to `True` to compute the spatial of average of the grid cells inside each polygon
- `time_avg` : set to `True` to create 30-year climatological means and anomalies/deltas relative to a reference period
- `ref_horizons` : reference horizons relative to which the anomalies will be calculated
- `ensemble_percentiles` : set to `True` to compute percentiles across ensemble members
- `output_folder` : the path to the output folder where Zarr and CSV files will be written

<div class="alert alert-success">
<strong>User input required below</strong> 
</div>

In [ ]:
### Change only the values on the right-hand side of the `=` signs. ###

# Region file (shapefile or GeoJSON)
polygon_file = "GIS-Data/NS_regional_admin_boundaries/NS_regional_admin_boundaries.shp"  # Nova Scotia admin regions example
# polygon_file = "GIS-Data/Maritimes_National_Parks/Maritimes_National_Parks_2026_02_04.shp"    # Maritime National Parks example
# polygon_file = "GIS-Data/watersheds/watersheds_southernQC.shp"                                # Quebec example

# Climate indicators of interest, grouped by temporal frequency. The variable IDs must exist in the table above.
variables = {
    "seasonal": ["prtot"],
    # "monthly":  ["tg_mean", "prx1day"],
    # "annual": ["tn_mean", "sn10mm", "pr1mm"],
    # ...
}

# Emission scenarios of interest
ssps = ["ssp370"]  # you can add other scenarios such as "ssp126", "ssp245", "ssp585"

# Average grid cells inside each polygon?
spatial_avg = True  # set this to False if you want a value per grid cell

# Average over 30-year periods and compute deltas?
time_avg = True  # set this to False if you need actual yearly data

# Reference horizons for the anomalies/deltas (used only if time_avg is True)
# Possible to have more than one : ["1981-2010", "1991-2020"]. The format must be "startyear-endyear" and the period must be 30 years long.
ref_horizons = ["1991-2020"]

# Calculate ensemble percentiles instead of keeping each member?
ensemble_percentiles = False  # for the example in Step 8, we keep individual ensemble members; set this to True if you want to calculate ensemble percentiles

# Where to save Zarr and CSV files?
# By default, the output folder is the same as the polygon_file filename (without extension) but you can change it to any path you want.
output_folder = "output/" + Path(polygon_file).stem

print("Polygon                                :", polygon_file)
print("Variables of interest (grouped by freq):", variables)
print("Scenarios                              :", ssps)
print("Calculate spatial averages?            :", spatial_avg)
print("Calculate 30-year averages and deltas? :", time_avg)
print("If deltas are calculated, reference(s) :", ref_horizons)
print("Calculate ensemble percentiles?        :", ensemble_percentiles)
print("Output dir                             :", output_folder)

### **Step 4 – Preview the region**

Plot the region defined in your shapefile (`polygon_file`) to visually confirm that you selected the correct file, and display the information contained in the shapefile as a table.

The subplot on the left shows the regions defined in the shapefile. The subplot on the right shows the gridded CanDCS-M6 data overlaid on the polygons, allowing you to check their spatial overlap.

Just run the cell below, no modification is needed.

In [ ]:
# simplify polygon geometries to avoid potential errors with subsetting and spatial averaging
gdf = simplify_shape(polygon_file)

gdf["colorfield"] = gdf.index

# empty plot
f, axs = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
# plot color
f = gdf.plot(ax=axs[0], column="colorfield", cmap="rainbow")
# plot boundary
f = gdf.boundary.plot(ax=axs[0], linewidth=0.25, color="k")
# plot data source grid to the background
with ProgressBar():
    smp_data = get_sample_data(gdf, data_source=data_source).squeeze().load()
smp_data.plot(ax=axs[1], alpha=0.4, add_colorbar=False, add_labels=False)

# # plot boundary
f = gdf.boundary.plot(ax=axs[1], linewidth=0.25, color="k")
f = axs[0].set_title("Preview of polygons")
f = axs[1].set_title(f"Polygons vs spatial extent of {data_source} data")
gdf.head()

Now, look at the table above where the shapefile information is displayed. Which column contains the names of the sub-regions (e.g., "Central", "Eastern" etc.)? Enter that column name below in `reg_field`.

This step is needed because the column names differ from one shapefile to another, so we must explicitly specify which column identifies the regions.
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>

In [ ]:
# name of the column in the shapefile that contains the region names
reg_field = "region"  # Nova Scotia admin regions example
# reg_field = 'NAME_E'   # Maritime National park example
# reg_field = 'label_en' # QC watersheds example

### **Step 5 – Extract the data, apply the analysis and save the results**

This is a long block of code but the good news is that you do not need to edit it. Just run it as is. The code will:
- find and load the CanDCS-M6 data from the THREDDS catalog,
- trim the data down to only the SSPs and variables you selected in Step 3,
- apply the analysis settings you selected in Step 3 (e.g., `time_avg`, `spatial_avg` etc.),
- and save the processed results as both a **Zarr** file and a **CSV** file.

A Zarr file is a format designed to efficiently store large, multi-dimensional datasets. The CSV file is useful if you prefer to explore the results in Excel.

Once the code has finished running, you will find both outputs in the `output` data folder. The saved Zarr file will also be used in the next step for data visualization. Since both file formats can be downloaded, you can easily transfer them to other platforms for further analysis.

In [ ]:
# Make sure things are Paths / lists
if isinstance(polygon_file, str):
    polygon_file = Path(polygon_file)

if isinstance(ssps, str):
    ssps = [ssps]

output_folder = Path(output_folder)
(output_folder / data_source / "zarr").mkdir(parents=True, exist_ok=True)
(output_folder / data_source / "csv").mkdir(parents=True, exist_ok=True)

url = pavics_url[data_source]["url"]
chunks = pavics_url[data_source]["chunks"]

ds = {}

with ProgressBar():
    for season, var_list in variables.items():
        if not var_list:
            continue

        print(f"\n=== {season} ===")

        # Find the appropriate dataset on THREDDS for this temporal grouping
        season_descriptor = None
        for dd in tds.crawl(url, depth=100):
            if data_source not in dd.name:
                continue
            # In CanDCS-M6, filenames usually start with the temporal grouping:
            # e.g. "seasonal_", "monthly_", "annual_"
            if dd.name.split("_")[0] == season:
                season_descriptor = dd
                break

        if season_descriptor is None:
            print(f"  No dataset found for temporal grouping '{season}'. Skipping.")
            continue

        # open the dataset with xarray
        print(f"Using dataset: {season_descriptor.name}")
        ds_raw = xr.open_dataset(
            season_descriptor.opendap_url(),
            chunks=chunks,
            decode_timedelta=False,
        )

        # set non-time variables as coordinates
        for vv in ds_raw.data_vars:
            if "time" not in ds_raw[vv].dims:
                ds_raw = ds_raw.assign_coords({vv: ds_raw[vv]})

        # Load coordinates in memory (helps with later operations)
        for cc in ds_raw.coords:
            ds_raw[cc] = ds_raw[cc].load()

        print(
            "Processing and saving the final datasets. This may take some time to run...\n"
        )
        season_dsets = []

        for ssp in ssps:
            print(f"Scenario: {ssp}")

            # In CanDCS-M6, variables are named like "ssp245_tx_mean"
            var_names = [v for v in ds_raw.data_vars if v.startswith(ssp)]
            if not var_names:
                print(f"    No variables starting with '{ssp}_' found. Skipping.")
                continue

            ds_tmp = ds_raw[var_names]

            # Rename variables to drop the ssp prefix: "ssp245_tx_mean" → "tx_mean"
            rename_map = {v: "_".join(v.split("_")[1:]) for v in var_names}
            ds_tmp = ds_tmp.rename(rename_map)

            # Keep only the variables requested by the user
            ds_tmp = ds_tmp[[v for v in ds_tmp.data_vars if v in var_list]]

            # Add a `scenario` coordinate
            ds_tmp = ds_tmp.assign_coords(scenario=ssp).expand_dims("scenario")

            season_dsets.append(ds_tmp)

        if not season_dsets:
            print(f"No data retained for {season}.")
            continue

        if len(season_dsets) > 1:
            ds_season = xr.concat(season_dsets, dim="scenario")
        else:
            ds_season = season_dsets[0]

        # Make sure `realization` is a string
        if "realization" in ds_season.coords:
            ds_season["realization"] = ds_season["realization"].astype(str)

        outfile_base = f"{season}_{data_source}"
        if ensemble_percentiles:
            outfile_base += "_ensemble_percentiles"
        else:
            outfile_base += "_members"

        # Spatial subset using the polygon
        print(f"Subsetting to polygon: {polygon_file.name}")

        # make sure longitude and latitude are CF-compliant so xscen can find them
        if "lon" in ds_season.coords and "lat" in ds_season.coords:
            ds_season["lon"].attrs.setdefault("standard_name", "longitude")
            ds_season["lon"].attrs.setdefault("units", "degrees_east")
            ds_season["lat"].attrs.setdefault("standard_name", "latitude")
            ds_season["lat"].attrs.setdefault("units", "degrees_north")
        else:
            raise ValueError(
                f"Could not find 'lon' and 'lat' coordinates in dataset. "
                f"Available coords: {list(ds_season.coords)}"
            )

        ds_season = subset(
            ds_season, method="shape", shape=simplify_shape(polygon_file), buffer=0.1
        )

        # Convert Kelvin → Celsius if needed
        for v in ds_season.data_vars:
            units = ds_season[v].attrs.get("units", "")
            if units.upper() in ["K", "KELVIN", "DEGK"]:
                print(f"Converting {v} from {units} to degC")
                ds_season[v] = convert_units_to(ds_season[v], "degC")
                ds_season[v].attrs["units"] = "degC"

        # Optional: 30-year climatologies and deltas
        if time_avg:
            print("Calculating 30-year means and deltas...")
            outfile = outfile_base + "_30yAvg"

            periods = [1951, 2100]  # full period; you can change if you want
            step = 10  # one 30-year climatology every 10 years

            # 30-year rolling means (climatologies)
            ds30 = climatological_op(
                ds_season,
                window=30,
                op="mean",
                min_periods=25,
                rename_variables=False,
                periods=periods,
            )

            # deltas relative to reference horizons
            datasets = [ds30]

            if ref_horizons:
                # allow both a single string or a list
                if isinstance(ref_horizons, str):
                    horizons = [ref_horizons]
                else:
                    horizons = list(ref_horizons)

                for horizon in horizons:
                    print(f"Computing deltas relative to {horizon}...")
                    ds_delta = compute_deltas(ds=ds30, reference_horizon=horizon)
                    datasets.append(ds_delta)
            else:
                print(
                    "No reference horizons provided in `ref_horizons`. Only 30-year climatologies (no deltas) will be computed."
                )

            # Merge climatologies and any deltas that were created
            ds_season = xr.merge(datasets)

            # Unstack dates
            ds_season = unstack_dates(ds_season, new_dim="temp_grp")
            ds_season = ds_season.isel(time=range(0, len(ds_season.time), step))

            # If 'horizon' has a season dimension, keep only one copy
            if "horizon" in ds_season and "temp_grp" in ds_season["horizon"].dims:
                ds_season["horizon"] = ds_season["horizon"].isel(temp_grp=0).squeeze()
        else:
            outfile = outfile_base
            ds_season = unstack_dates(ds_season, new_dim="temp_grp")

        # Optional: spatial averaging over the polygon
        if spatial_avg:
            print("Calculating spatial averages over the polygon…")
            outfile += f"_spatial-avg_{polygon_file.stem}"

            poly = simplify_shape(polygon_file)
            region = dict(name=polygon_file.stem, method="shape", shape=poly)

            ny = len(ds_season.lat)
            nx = len(ds_season.lon)
            wgtfile = polygon_file.parent.joinpath(
                f"{data_source}_{nx}x{ny}_{polygon_file.stem}_weights.nc"
            )

            if "rlon" in ds_season.dims:
                if "crs" in ds_season.data_vars or "crs" in ds_season.coords:
                    ds_season = ds_season.drop_vars("crs")

            ds_season = spatial_mean(
                ds_season,
                method="xesmf",
                region=region,
                kwargs={
                    "filename": wgtfile,
                    "skipna": True,
                    "reuse_weights": wgtfile.exists(),
                },
            )

        # Clean up bounds-like coordinates
        ds_season = ds_season.drop_vars(
            [c for c in ds_season.coords if "bounds" in c],
            errors="ignore",
        ).squeeze()

        # Make sure scenario & season exist as dimensions
        for dim in ["scenario", "temp_grp"]:
            if dim not in ds_season.dims:
                ds_season = ds_season.expand_dims(dim)

        # Optional: compute ensemble percentiles
        if ensemble_percentiles:
            print("Computing ensemble percentiles…")
            ds_season = xens.ensemble_stats(
                ds_season,
                {"ensemble_percentiles": {"split": False}},
            )

        # Save to Zarr and reopen
        outzarr = output_folder / data_source / "zarr" / f"{outfile}.zarr"
        if outzarr.exists():
            # check if new variables have been added to the user list
            ds_old = xr.open_zarr(outzarr)
            update_flag = any([v not in ds_old.data_vars for v in ds_season.data_vars])
            update_flag = update_flag or any(
                [
                    scen not in ds_old.scenario.values
                    for scen in ds_season.scenario.values
                ]
            )
            if update_flag:
                print(
                    f"New variables or emissions scenarios added.\n{outzarr.name} exists but new user variables have been added ... re-exporting."
                )

        if not outzarr.exists() or update_flag:
            if outzarr.exists():
                shutil.rmtree(outzarr)
            print(f"Saving to Zarr: {outzarr}")
            with Client(**dask_kwargs) as c:
                display(c)  # display the Client dashboard link
                outchunks = {}
                for d in ds_season.dims:
                    if d in pavics_url[data_source]["chunks"]:
                        outchunks[d] = pavics_url[data_source]["chunks"][d]
                    elif d == "geom":
                        outchunks[d] = 250
                    else:
                        outchunks[d] = 1
                save_to_zarr(
                    ds_season.chunk(outchunks),
                    outzarr.with_suffix(".tmp.zarr"),
                    mode="w",
                )

            shutil.move(outzarr.with_suffix(".tmp.zarr"), outzarr)
        del ds_season

        print("Re-opening Zarr…")
        ds_saved = xr.open_zarr(outzarr, decode_timedelta=False)
        ds[season] = ds_saved

        # Export each temporal grouping dataset to a CSV
        outcsv = output_folder / data_source / "csv" / f"{outzarr.stem}.csv"
        outmetacsv = (
            output_folder / data_source / "csv" / f"{outzarr.stem}.metadata.csv"
        )
        with ProgressBar():
            print(f"\nExporting to CSV : {outcsv}")
            outcsv.parent.mkdir(exist_ok=True)
            ddf = ds_saved.unify_chunks().to_dask_dataframe()
            ddf.to_csv(outcsv, single_file=True)

            # save metadata CSV
            print(f"Saving the CSV containing the metadata : {outmetacsv}")
            meta = {}
            for v in ds_saved.data_vars:
                meta[v] = {}
                for k in ["units", "long_name", "description"]:
                    if k in ds_saved[v].attrs:
                        meta[v][k] = ds_saved[v].attrs[k]
            meta_df = pd.DataFrame.from_dict(meta).T
            meta_df.to_csv(outmetacsv)

    print("\nAll done. The final datasets are stored in the dictionary `ds`.")
    print("Keys:", list(ds.keys()))

### **Step 6 – Visualize the processed data**

Let’s now visualize the indicator data processed in the previous steps. In this exercise, you will create a grouped box plot comparing the historical and future seasonal patterns of a selected indicator for one sub-region.

In this part, we will:

1. open the newly generated output dataset from the Zarr file: ***\***CanDCS-M6_members_30yAvg_spatial-avg**\**.zarr**, 
2. define the plotting settings: which variable, sub-region, historical and future periods to plot, 
3. create a grouped box plot and save it in the `figures` folder.

No modification is needed in the cell below. Once you run it, it will open the newly generated dataset and show its contents.

In [ ]:
temp_group = "seasonal"
# find matching patterns
zarr = list(
    output_folder.joinpath(data_source, "zarr").glob(
        f"{temp_group}_*members_30yAvg_spatial-avg_*.zarr"
    )
)
if len(zarr) != 1:
    print(
        f"found {len(zarr)} datasets - {zarr} .. expected one. Make sure to generate a new output with user specifications described above."
    )
else:
    ds_seasonal = xr.open_zarr(zarr[0])
ds_seasonal

The output dataset is an xarray.Dataset, which organizes multi-dimensional data using labeled dimensions, coordinates, and data variables. In this example, the dataset has five dimensions, several coordinates that provide additional information about the data, and two data variables. Here is what each of them represents:

**Dimensions**:
  - `geom` (59): sub-regions inside the polygon file (e.g. administrative regions)
  - `time` (13): start years of each 30-year climatological period (e.g. `"1951-01-01"` for `"1951-1980"`)
  - `scenario` (1): the emission scenario (here: `ssp370`)
  - `temp_grp` (4): temporal grouping : here, the four seasons `MAM` (spring), `JJA` (summer), `SON` (fall), `DJF` (winter)
  - `realization` (24): the individual ensemble members (one per each model/member combination)

**Coordinates**:
  - `lat`, `lon`: representative latitude and longitude coordinates for each sub-region
  - `horizon`: label describing each 30-year period separated by 10 years (e.g. `"1951-1980"`, `"1961-1990"`, ..., `"2071-2100"`)
  - & other coordinates.

**Data variables**:
  - `prtot`: 30-year mean seasonal total precipitation for each combination of (geom, time, scenario, temp_grp, realization)
  - `prtot_delta_1991_2020`: the change in `prtot` relative to the 1991–2020 reference period

Let’s first list the variables, sub-regions, and periods available in this dataset. You will then choose one item from each list for plotting.

In [ ]:
print("Available variables to choose from:", ", ".join(ds_seasonal.data_vars))
print(
    "\nAvailable sub-regions to choose from:",
    ", ".join(map(str, ds_seasonal[reg_field].values)),
)
print(
    "\nAvailable periods to choose from:",
    ", ".join(map(str, ds_seasonal["horizon"].values)),
)

Next, using the lists above, specify the variable, sub-region, and historical and future periods you want to compare.
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>

In [ ]:
# === Change only the values on the right-hand side of the `=` signs. ====
analyze_variable = "prtot"

analyze_region = "Central"  # for NS admin boundary example, choose one of the 4 regions (e.g. "Central", "Eastern", "Northern", "Western")
# analyze_region = "CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA"  # for Maritime National Parks example
# analyze_region = "Central Ottawa - Dumoine"                      # for QC watersheds example

historical_period = "1991-2020"
future_period = "2071-2100"

Finally, create a grouped box plot to compare the seasonal distribution of the selected indicator between the historical and future periods. Run the cell below to generate and save the figure.

In [ ]:
from pathlib import Path

# import the plotting libraries
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# select the sub-region
regional_ds = ds_seasonal.sel(geom=ds_seasonal[reg_field] == analyze_region)

# subset the selected variable and periods, then remove the length-1 time dimension
historical_data = (
    regional_ds[analyze_variable]
    .sel(time=regional_ds["horizon"] == historical_period)
    .squeeze("time")
)

future_data = (
    regional_ds[analyze_variable]
    .sel(time=regional_ds["horizon"] == future_period)
    .squeeze("time")
)

# build a long-format dataframe for plotting
rows = []
for season in ds_seasonal["temp_grp"].values:
    rows.extend(
        (season, "Historical", val)
        for val in historical_data.sel(temp_grp=season).values.ravel()
    )
    rows.extend(
        (season, "Future", val)
        for val in future_data.sel(temp_grp=season).values.ravel()
    )

df_long = pd.DataFrame(rows, columns=["Season", "Period", "Value"])

# define custom color palettes for boxes and points
palette_boxes = {"Historical": "#1f77b4", "Future": "#ff7f0e"}
palette_points = {
    "Historical": "#5fa6d9",
    "Future": "#f2a65a",
}  # slightly different tones

# figure creation
plt.figure(figsize=(10, 6))

# create the boxplot using the Seaborn library
sns.boxplot(x="Season", y="Value", hue="Period", data=df_long, palette=palette_boxes)

# overlay the individual data points
sns.stripplot(
    x="Season",
    y="Value",
    hue="Period",
    data=df_long,
    dodge=True,
    jitter=False,
    size=5,
    palette=palette_points,
)

# remove duplicate legends caused by boxplot + stripplot
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title="Period")

# get variable unit and long name for labels
unit = historical_data.attrs.get("units", "Unknown")
long_name = (
    regional_ds[analyze_variable]
    .attrs.get("long_name", analyze_variable)
    .rstrip(".")
    .capitalize()
)

# add labels and title
plt.xlabel("Season")
plt.ylabel(f"{analyze_variable} ({unit})")
plt.title(f"{long_name} - {analyze_region}: {historical_period} vs {future_period}")
plt.tight_layout()

# save figure
fig_folder = Path("figures")
fig_folder.mkdir(exist_ok=True)
fig_path = (
    fig_folder
    / f"boxplot_{analyze_region}_{analyze_variable}_{data_source}_{historical_period}-{future_period}.png"
)
plt.savefig(fig_path)
print(f"Figure saved to {fig_path}")

# display the figure
plt.show()